# Transfer Learning with ResNet50 on TF Flowers Dataset
This notebook compares `linear_probe` and `stage5` with three seeds each.

In [ ]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# Check TF and GPU
print("TF version:", tf.__version__)
print("All physical devices:")
print(tf.config.list_physical_devices())

print("\nGPU devices:")
print(tf.config.list_physical_devices("GPU"))


In [ ]:
# Download flower data
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
tf.keras.utils.get_file(origin=dataset_url, fname="flower_photos", untar=True)

data_dir = os.path.expanduser("~/.keras/datasets/flower_photos/flower_photos")

print("Data folder:", data_dir)
print(os.listdir(data_dir))

In [ ]:
# Build train/val sets
img_size = (224, 224)
batch_size = 128

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

In [ ]:
# Show class names
class_names = train_ds.class_names
print("Class names:", class_names)

In [ ]:
# Augmentation and stable input pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

# Do NOT preprocess in tf.data; preprocessing is done inside the model.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(1)
val_ds   = val_ds.prefetch(1)

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

@tf.keras.utils.register_keras_serializable()
class SmoothedSparseCategoricalCrossentropy(tf.keras.losses.Loss):
    def __init__(self, num_classes=5, label_smoothing=0.1, name="smoothed_sparse_categorical_crossentropy"):
        super().__init__(name=name)
        self.num_classes = num_classes
        self.label_smoothing = label_smoothing
        self.inner = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true = tf.one_hot(y_true, depth=self.num_classes)
        return self.inner(y_true, y_pred)

    def get_config(self):
        return {
            "num_classes": self.num_classes,
            "label_smoothing": self.label_smoothing,
            "name": self.name,
        }

def build_model(train_stage: str):
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # 1) Freeze everything first
    for layer in base_model.layers:
        layer.trainable = False

    # 2) Unfreeze by stage
    if train_stage == "stage5":
        for layer in base_model.layers:
            if layer.name.startswith("conv5_"):
                layer.trainable = True
    elif train_stage == "linear_probe":
        pass
    else:
        raise ValueError(f"Unknown train_stage: {train_stage}")

    # 3) Always freeze BatchNorm for small datasets
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)                 # preprocess ONLY here
    x = base_model(x, training=False)       # keep BN behavior stable
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(5, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=3e-6,
            weight_decay=1e-5
        ),
        loss=SmoothedSparseCategoricalCrossentropy(num_classes=5, label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model

In [ ]:
import os, json, time, random
import numpy as np
import tensorflow as tf

os.makedirs("Plot", exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def count_trainable_params(model):
    return int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))

def run_one(train_stage: str, seed: int, epochs: int = 15):
    set_seed(seed)
    tf.keras.backend.clear_session()

    model = build_model(train_stage=train_stage)
    params = count_trainable_params(model)

    ckpt_path = f"Plot/best_{train_stage}_seed{seed}.keras"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True, mode="min"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True, mode="min"
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7
        ),
    ]

    t0 = time.time()
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )
    t1 = time.time()

    best_val_acc  = float(np.max(hist.history["val_accuracy"]))
    best_val_loss = float(np.min(hist.history["val_loss"]))
    return {
        "stage": train_stage,
        "seed": seed,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "epochs_ran": len(hist.history["loss"]),
        "train_seconds": (t1 - t0),
        "trainable_params": params,
        "ckpt": ckpt_path
    }

seeds = [1, 2, 3]
settings = ["linear_probe", "stage5"]

runs = []
for st in settings:
    for s in seeds:
        runs.append(run_one(st, s, epochs=15))

summary = {}
for st in settings:
    sub = [r for r in runs if r["stage"] == st]
    accs = [r["best_val_acc"] for r in sub]
    times = [r["train_seconds"] for r in sub]
    params = [r["trainable_params"] for r in sub]
    summary[st] = {
        "best_val_acc_mean": float(np.mean(accs)),
        "best_val_acc_std": float(np.std(accs, ddof=1)),
        "train_sec_mean": float(np.mean(times)),
        "trainable_params": int(np.mean(params)),
    }

print(json.dumps(summary, indent=2))
with open("results_summary.json", "w") as f:
    json.dump({"runs": runs, "summary": summary}, f, indent=2)